# Laboratorio de clasificación 3

|                |   |
:----------------|---|
| **Nombre**     |  Claudio Alejo Encarnación Martínez |
| **Fecha**      |  12 octubre 2026 |
| **Expediente** |  750597 | 

Una regresión logística simple toma **P** predictores y usa una función no lineal para convertir la respuesta binaria **Y** en probabilidades entre $0$ y $1$.

La visualización de este escenario es puntos separados por una línea recta.

<center><img src="https://wirelesspi.com/wp-content/uploads/2022/08/figure-logistic-regression-decision-boundary-2-570x550.png"></center>

Para casos donde los datos no son linealmente separables usamos el truco del kernel para mapear los datos a un espacio de Hilbert.

Imagina un escenario donde tienes dos predictores $x_1, x_2$ y la salida de estos predictores se comporta de esta manera:

| **$x_1$** | **$x_2$** | Salida |
:-----------|-----------|--------|
| 0         | 0         | 0      |
| 0         | 1         | 1      |
| 1         | 0         | 1      |
| 1         | 1         | 0      |


Imagínalo como el interruptos de luz de un foco en una escalera. Se puede prender y apagar desde arriba y desde abajo.

<center><img src="https://www.researchgate.net/profile/Brian-Spears/publication/322048911/figure/fig4/AS:667820685807618@1536232267250/The-exclusive-or-XOR-function-is-a-nonlinear-function-that-returns-0-when-its-two.png"></center>

¿Cómo podemos separar estos datos usando sólo regresión logística? Sin importar cómo usemos una sola línea, nunca se van a poder separar todos los casos, por lo que tendríamos situaciones donde siempre habría error. 

Solución: ¿Por qué nos estamos limitando a una sola línea?

<center><img src="https://pabloinsente.github.io/assets/post-5/XOR-linear.png"></center>

La regresión logística sola es un método de aprendizaje estadístico convexo. Combinar dos regresiones logísticas para clasificar datos no linealmente separables convierte el problema a uno no convexo. ¿Por qué?

- Una regresión logística simple es convexa porque el modelo es lineal en los parámetros y la función de pérdida (log-loss) es convexa. Al combinar dos regresiones logísticas, por ejemplo en una capa oculta con activaciones sigmoideas y una capa de salida también sigmoidea, la salida final depende de productos y composiciones de funciones no lineales. Esa composición hace que la pérdida resultante ya no sea convexa en el conjunto de todos los pesos; aparecen múltiples mínimos locales, mesetas y puntos silla.

¿Qué significa que una función sea convexa?

- Una función convexa es aquella en la que, para cualesquiera dos puntos en su dominio, el segmento de línea que los une se encuentra por encima o sobre la gráfica de la función. Matemáticamente, una función f es convexa si para todos (x, y) en su dominio y para todo \(\lambda \in [0,1]\) se cumple:


¿Por qué nos importa que una función sea convexa o no? ¿Qué impacto tiene para los resultados?

- En optimización convexa, cualquier mínimo local es mínimo global. Esto garantiza que métodos como descenso de gradiente converjan al óptimo global (bajo condiciones adecuadas), sin depender fuertemente de la inicialización. En problemas no convexos, como las redes neuronales, pueden existir múltiples mínimos locales y puntos silla; el resultado depende de la inicialización, la tasa de aprendizaje, el optimizador, etc. Por eso se usan múltiples reinicios, validación, early stopping y técnicas estocásticas para encontrar buenas soluciones, aunque no se garantice el óptimo global.

## Ejercicio 1

Utilizaremos el clasificador `MLPClassifier` de la librería `sklearn`.

Entra a la documentación y describe cómo funciona este objeto (qué entradas necesita, qué salidas da, descripción de sus hiperparámetros).

- MLPClassifier de sklearn implementa un perceptrón multicapa para clasificación.

Entradas:
- X: array-like de forma (n_samples, n_features).
- y: array-like de forma (n_samples,) con etiquetas de clase.

Salidas:
- `predict(X)`: devuelve las etiquetas predichas.
- `predict_proba(X)`: devuelve probabilidades por clase.
- `predict_log_proba(X)`: log-probabilidades.

Atributos tras el ajuste: classes_, loss_, coefs_, intercepts_, n_iter_, out_activation_.

Hiperparámetros principales:
- `hidden_layer_sizes`: tupla con el número de neuronas por capa oculta. Default (100,).
- `activation`: función de activación: 'identity', 'logistic', 'tanh', 'relu'. Default 'relu'.
- `solver`: algoritmo de optimización: 'lbfgs', 'sgd', 'adam'. Default 'adam'.
- `alpha`: regularización L2. Default 0.0001.
- `batch_size`: tamaño de lote para 'sgd' y 'adam'. Default 'auto'.
- `learning_rate`: esquema de tasa de aprendizaje para 'sgd': 'constant', 'invscaling', 'adaptive'.
- `learning_rate_init`: tasa inicial. Default 0.001.
- `max_iter`: número máximo de iteraciones. Default 200.
- `shuffle`: mezclar muestras en cada iteración. Default True.
- `random_state`: semilla para reproducibilidad.
- `tol`: tolerancia para criterio de parada.
- `early_stopping`: si es True, usa validación interna para detener el entrenamiento.
- `validation_fraction`: proporción de datos para validación si early_stopping=True.
- `n_iter_no_change`: paciencia para early stopping.
- `beta_1`, `beta_2`, `epsilon`: parámetros de Adam.
- `momentum`, `nesterovs_momentum`: para SGD.

## Ejercicio 2

Utiliza el objeto de perceptrón multicapa para clasificar si un tumor es maligno o benigno con el dataset `Brain Tumor Dataset`. Utiliza un cross-validation de train-test de 70/30. Sigue todos los pasos de EDA y limpieza vistos.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_brain_tumor
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, RocCurveDisplay
)

# 1. Carga de datos
df = load_brain_tumor(as_frame=True).frame.copy()

# 2. EDA básico
print("Primeras filas:")
display(df.head())
print("\nInformación:")
df.info()
print("\nDescripción estadística:")
display(df.describe(include='all'))
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nDuplicados:")
print(df.duplicated().sum())

# 3. Limpieza
# Ajusta target_col e id_col según tu dataset
target_col = 'diagnosis'   # Ejemplo: 'diagnosis', 'malignant', 'Tumor_Type', etc.
id_col = 'Patient_ID'      # Si no existe, elimina esta línea

df = df.drop_duplicates()

# Si el target tiene nulos, eliminamos esas filas
df = df.dropna(subset=[target_col])

# Mapear target a binario: benigno=0, maligno=1
# Ajusta los nombres según tus clases reales
df[target_col] = df[target_col].map({'benign': 0, 'malignant': 1})

# Si quedan nulos en el target después del mapeo, revisar valores únicos
print("Clases:", df[target_col].unique())

# Separar variables
X = df.drop(columns=[target_col] + ([id_col] if id_col in df.columns else []))
y = df[target_col]

# Identificar columnas numéricas y categóricas
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# 4. Preprocesamiento
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# 5. Split 70/30 estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# 6. Modelo MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    random_state=42
)

clf = Pipeline(steps=[
    ('prep', preprocessor),
    ('mlp', mlp)
])

clf.fit(X_train, y_train)

# 7. Predicciones y métricas
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

# Curva ROC
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("Curva ROC - MLP")
plt.show()

ImportError: cannot import name 'load_brain_tumor' from 'sklearn.datasets' (/opt/miniconda3/lib/python3.13/site-packages/sklearn/datasets/__init__.py)

- **respuesta aquí**

¿Qué métrico crees que sea mejor para este ejercicio? Escoge los 3 métricos que crees más apropiados para esta predicción, argumenta en pro y en contra del top 3. Elige uno para optimizar (justifica tu respuesta).

## Ejercicio 3

Modifica los hiperparámetros para elevar el métrico elegido. Intenta con 10 sets de hiperparámetros distintos. Crea una tabla que muestre los resultados.

- **respuesta aquí**